# 03. Baseline Model

Continuing from `cleaned_preprocessed_data.csv` (created in 02_preprocessing.ipynb). That file already has missing values imputed, categorical fields encoded, and numeric features scaled. All `fit` on the train window only, with a `split` column marking which rows are `train` / `test` / `unused`.

Here we train a Linear Regression as the first baseline model, evaluate it with R² on the test set, and record the results so later, more complex models can be compared against this number.


In [15]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

RANDOM_STATE = 42


## 1. Load cleaned dataset

Load `cleaned_preprocessed_data.csv` produced in 02_preprocessing.ipynb. The `split` column marks which rows belong to the train window and which belong to the test month (`unused` rows fall outside the best_X window chosen in week 3 and are excluded here).


In [16]:
cleaned_df = pd.read_csv("cleaned_preprocessed_data.csv", low_memory=False)
print(cleaned_df.shape)
cleaned_df["split"].value_counts()


(141988, 115)


split
train    129964
test      12024
Name: count, dtype: int64

## 2. Separate features / target using the `split` column

`ClosePrice` (target), `date`, `year_month`, and `split` are metadata columns, not model features, so they're excluded from X. Using `split` here (instead of re-deriving months) guarantees the exact same train/test rows used for the RandomForest comparison in 02_preprocessing.ipynb.


In [17]:
target = "ClosePrice"
meta_cols = [target, "date", "year_month", "split"]

train_df = cleaned_df[cleaned_df["split"] == "train"].copy()
test_df = cleaned_df[cleaned_df["split"] == "test"].copy()

feature_cols = [c for c in cleaned_df.columns if c not in meta_cols]

X_train = train_df[feature_cols]
y_train = train_df[target]

X_test = test_df[feature_cols]
y_test = test_df[target]

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")


X_train: (129964, 111) | X_test: (12024, 111)


## 3. Train baseline model (Linear Regression)

Since `cleaned_preprocessed_data.csv` was already imputed, encoded, and scaled in 02_preprocessing.ipynb, no additional preprocessing pipeline is needed here. Linear Regression is fit directly on the numeric feature matrix.


In [18]:
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)


LinearRegression()

## 4. Evaluate on the test set (R^2)

R^2 is the primary metric requested for this baseline. MAE / RMSE are also recorded alongside it so this baseline can be compared directly against the RandomForest numbers from 02_preprocessing.ipynb and against future models.


In [19]:
y_pred = baseline_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))  # squared=False was removed in newer sklearn, so use sqrt instead

print(f"Baseline Linear Regression  \n R2: {r2:.4f} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f}")


Baseline Linear Regression  
 R2: 0.3365 | MAE: 547,257 | RMSE: 1,366,886


## 5. Record baseline results

Save the baseline metrics to a CSV so later weeks (with more complex models) can compare against this number without re-running this notebook.


In [20]:
baseline_results = pd.DataFrame([{
    "model": "LinearRegression",
    "n_train_rows": len(X_train),
    "n_test_rows": len(X_test),
    "R2": r2,
    "MAE": mae,
    "RMSE": rmse,
}])

baseline_results.to_csv("baseline_model_results.csv", index=False)
print("Saved baseline_model_results.csv")
baseline_results


Saved baseline_model_results.csv


,model,n_train_rows,n_test_rows,R2,MAE,RMSE
0,LinearRegression,129964,12024,0.336487,547257.189138,1.366886e+06


In [21]:
print(f"train rows: {len(X_train)}, feature columns: {X_train.shape[1]}")

train rows: 129964, feature columns: 111


## Summary

- Baseline model: `LinearRegression` (no hyperparameters tuned. this is intentionally the simplest possible model)
- Trained on the same train window (`split == "train"`) and evaluated on the same test month (`split == "test"`) established in 02_preprocessing.ipynb, so results are directly comparable to future models
- Primary metric: R^2 on the test set (see cell 9 output above); MAE / RMSE recorded alongside for reference
- Deliverables: `baseline_model_results.csv`, this notebook (`03_baseline_model.ipynb`)
